# 03 — K-Means regimes and feature engineering
K-Means tạo nhãn phụ trạng thái HVAC. Clusterer chỉ fit trên temporal training set; validation/test chỉ được transform.

In [ ]:
from pathlib import Path
import sys, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
ROOT = Path.cwd().resolve(); ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path[:0] = [str(ROOT / 'src'), str(ROOT)]
from cooling_load.config import load_config
from cooling_load.io import read_frame
from cooling_load.pipeline import run_feature_engineering
from cooling_load.splitting import temporal_holdout
from cooling_load.clustering import OperatingRegimeClusterer
config = load_config(ROOT / 'configs/base.yaml')
clean = read_frame(config.path('interim_data'))
featured = run_feature_engineering(config, clean)
train, validation, test = temporal_holdout(featured, config.data.timestamp_column)


In [ ]:
clusterer = OperatingRegimeClusterer(config.clustering.features, config.clustering.n_clusters)
train_clustered = clusterer.fit_transform(train, 'cooling_load_kwh_lag_3')
validation_clustered = clusterer.transform(validation)
display(train_clustered.groupby('operating_regime')[list(config.clustering.features)].median())


In [ ]:
sns.countplot(data=train_clustered, x='operating_regime', order=['low_load','normal_load','high_load','peak_load']); plt.xticks(rotation=25); plt.title('Training operating regimes'); plt.show()
display(featured.filter(regex='lag_|mean_|std_|temperature_delta|thermal_load').head())
print('Feature dataset:', config.path('feature_data'))
